# DANFront - Second Military Survey (2KF) tiles

Downloads the 89 GB GeoTIFF archive from SharePoint into **Google Drive**, turns the sheets into map tiles
batch by batch and uploads them to the Cloudflare R2 bucket (`danfront/tiles-2kf/`). Nothing is stored on your own computer.

**Before you start**

1. Cloudflare dashboard > R2 > *Manage API Tokens* > *Create API Token*: permission *Object Read & Write*, bucket `danfront`.
2. In Colab, open the key icon (Secrets) in the left bar and add three secrets, with *Notebook access* switched on:
   `R2_ACCESS_KEY_ID`, `R2_SECRET_ACCESS_KEY`, `R2_ENDPOINT` (the `https://<account-id>.r2.cloudflarestorage.com` address).

**How to use**: run the cells from top to bottom. Every step can be run again: finished download parts and finished
sheets are skipped. If the session disconnects, reconnect and run cells 1 to 4 again, then continue with cell 7.
Keep the browser tab open while it works.

## 1. Setup (about 1 minute)
Mounts Google Drive and installs the tools. Approve the Drive permission window.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os
WORK  = '/content/drive/MyDrive/danfront_2kf'   # on Drive: archive parts, list of finished sheets
LOCAL = '/content/work'                          # on the Colab disk: extracted sheets and tiles (temporary)
os.makedirs(WORK, exist_ok=True); os.makedirs(LOCAL, exist_ok=True)

!pip -q install rasterio pyproj
!command -v 7z >/dev/null || apt-get -qq install -y p7zip-full >/dev/null
!command -v rclone >/dev/null || (curl -s https://rclone.org/install.sh | bash >/dev/null 2>&1)
!7z | sed -n 2p; rclone version | head -1; df -h /content | tail -1

## 2. Load the scripts
Nothing to change here.

In [ ]:
%%writefile /content/work/tiles_2kf_batch.py
"""
tiles_2kf_batch.py - cut XYZ tiles from many Second Military Survey (2KF) GeoTIFFs at once,
all sheets merged into one tile folder. Used by colab_2kf.ipynb, also runnable by hand.

Input: GeoTIFFs exported with qgis_batch_export.py (Cassini-Soldner, origin St. Stephen's
Cathedral Vienna). Each sheet is clipped automatically to its map frame:
the sheets are 9600 x 6400 Vienna Klafter (18206.24 x 12137.50 m) cells of a regular grid in
that Cassini system, with Vienna in the middle of a cell, i.e. cell (i, j) covers
  X in [(i-0.5)*W, (i+0.5)*W],  Y in [(j-0.5)*H, (j+0.5)*H].
(Checked on 32-50Cas: the nominal cell matches the drawn frame within ~25 m.)
A sheet whose image does not cover most of one cell is tiled unclipped and reported as
"irregular"; files without a CRS are skipped.

Usage:
  python3 tiles_2kf_batch.py <tiles_out_dir> <sheet1.tif> [<sheet2.tif> ...] [--zoom 6,14] [--workers 2]
                             [--bbox lonmin,latmin,lonmax,latmax]

Prints one JSON line per sheet (status, cell, tile bounds) and writes the list of tile files
it created or changed to <tiles_out_dir>/../changed_tiles.txt (relative paths, for
`rclone copy --files-from`). Tiles shared by neighbouring sheets are composited, with a
file lock so parallel workers don't clash.
"""
import argparse, fcntl, json, math, os, sys
from multiprocessing import Pool
import numpy as np
import rasterio
from rasterio.transform import Affine
from rasterio.warp import reproject, transform as warp_transform, Resampling
from rasterio.windows import from_bounds
from PIL import Image

R = 20037508.342789244
KLAFTER = 1.89648384
CELL_W, CELL_H = 9600 * KLAFTER, 6400 * KLAFTER
CASSINI_VIENNA = "+proj=cass +lat_0=48.208772 +lon_0=16.374242 +a=6376130 +rf=310 +units=m +no_defs"


def best_cell(b):
    """Grid cell with the largest overlap with image bounds b; returns (i, j, cell_bounds, covered_fraction_of_cell)."""
    best = None
    ci, cj = round((b.left + b.right) / 2 / CELL_W), round((b.bottom + b.top) / 2 / CELL_H)
    for i in (ci - 1, ci, ci + 1):
        for j in (cj - 1, cj, cj + 1):
            x0, x1, y0, y1 = (i - .5) * CELL_W, (i + .5) * CELL_W, (j - .5) * CELL_H, (j + .5) * CELL_H
            ox = max(0, min(b.right, x1) - max(b.left, x0)); oy = max(0, min(b.top, y1) - max(b.bottom, y0))
            frac = ox * oy / (CELL_W * CELL_H)
            if best is None or frac > best[3]:
                best = (i, j, (x0, y0, x1, y1), frac)
    return best


def composite_save(path, dst, lock_path):
    new = Image.fromarray(dst.transpose(1, 2, 0), "RGBA")
    with open(lock_path, "w") as lf:
        fcntl.flock(lf, fcntl.LOCK_EX)
        if os.path.exists(path):
            old = Image.open(path).convert("RGBA")
            old.alpha_composite(new); new = old
        os.makedirs(os.path.dirname(path), exist_ok=True)
        new.save(path, optimize=True)


def process(job):
    tif, out_dir, zmin, zmax, bbox = job
    sheet = os.path.splitext(os.path.basename(tif))[0]
    res = dict(sheet=sheet, status="", tiles=[], bounds={})
    try:
        with rasterio.open(tif) as ds:
            if ds.crs is None:
                res["status"] = "skipped: no CRS (RAW sheet)"; return res
            cas = rasterio.crs.CRS.from_proj4(CASSINI_VIENNA)
            same = abs(ds.crs.to_dict().get("lat_0", 0) - 48.208772) < 1e-6 and ds.crs.to_dict().get("proj") == "cass"
            if not same:
                res["status"] = f"skipped: unexpected CRS {ds.crs.to_proj4()[:60]}"; return res
            i, j, (x0, y0, x1, y1), frac = best_cell(ds.bounds)
            res["cell"] = [i, j]; res["cell_coverage"] = round(frac, 3)
            if frac >= 0.9:
                win = from_bounds(x0, y0, x1, y1, ds.transform).round_offsets().round_lengths()
                win = win.intersection(rasterio.windows.Window(0, 0, ds.width, ds.height))
                res["status"] = "ok"
            else:
                win = rasterio.windows.Window(0, 0, ds.width, ds.height)
                res["status"] = "ok, irregular (not clipped)"
            data = ds.read([1, 2, 3], window=win)
            src_tf = ds.window_transform(win); src_crs = ds.crs
        rgba = np.zeros((4,) + data.shape[1:], np.uint8)
        rgba[:3] = data; rgba[3] = np.where(data.any(axis=0), 255, 0); del data
        H, W = rgba.shape[1:]
        n = 20
        edge = ([(W*k/n, 0) for k in range(n+1)] + [(W, H*k/n) for k in range(n+1)] + [(W*k/n, H) for k in range(n+1)] + [(0, H*k/n) for k in range(n+1)])
        xs, ys = zip(*[src_tf * p for p in edge])
        mx, my = warp_transform(src_crs, "EPSG:3857", xs, ys)
        minX, maxX, minY, maxY = min(mx), max(mx), min(my), max(my)
        if bbox:
            lon, lat = warp_transform("EPSG:3857", "EPSG:4326", [minX, maxX], [minY, maxY])
            if lon[1] < bbox[0] or lon[0] > bbox[2] or lat[1] < bbox[1] or lat[0] > bbox[3]:
                res["status"] = "skipped: outside bbox"; return res
        lock_path = os.path.join(os.path.dirname(os.path.abspath(out_dir)), ".tiles.lock")
        for z in range(zmin, zmax + 1):
            size = 2 * R / 2 ** z
            xa, xb = int((minX + R) // size), int((maxX + R) // size)
            ya, yb = int((R - maxY) // size), int((R - minY) // size)
            res["bounds"][z] = dict(xMin=xa, xMax=xb, yMin=ya, yMax=yb)
            for x in range(xa, xb + 1):
                for y in range(ya, yb + 1):
                    dst = np.zeros((4, 256, 256), np.uint8)
                    dtf = Affine(size / 256, 0, -R + x * size, 0, -size / 256, R - y * size)
                    reproject(rgba, dst, src_transform=src_tf, src_crs=src_crs, dst_transform=dtf, dst_crs="EPSG:3857",
                              resampling=Resampling.average if z < 14 else Resampling.bilinear)
                    if dst[3].max() == 0:
                        continue
                    rel = f"{z}/{x}/{y}.png"
                    composite_save(os.path.join(out_dir, rel), dst, lock_path)
                    res["tiles"].append(rel)
    except Exception as e:
        res["status"] = f"ERROR: {type(e).__name__}: {e}"
    return res


def run(tifs, out_dir, zmin=6, zmax=14, workers=2, bbox=None):
    """Process the given GeoTIFFs; returns the list of per-sheet result dicts."""
    os.makedirs(out_dir, exist_ok=True)
    jobs = [(t, out_dir, zmin, zmax, bbox) for t in tifs]
    if workers > 1 and len(jobs) > 1:
        with Pool(workers) as p:
            results = p.map(process, jobs, chunksize=1)
    else:
        results = [process(j) for j in jobs]
    changed = sorted({t for r in results for t in r["tiles"]})
    with open(os.path.join(os.path.dirname(os.path.abspath(out_dir)), "changed_tiles.txt"), "w") as f:
        f.write("\n".join(changed) + ("\n" if changed else ""))
    return results


def merge_bounds(total, b):
    for z, v in b.items():
        z = int(z); t = total.setdefault(z, dict(v))
        t.update(xMin=min(t["xMin"], v["xMin"]), xMax=max(t["xMax"], v["xMax"]), yMin=min(t["yMin"], v["yMin"]), yMax=max(t["yMax"], v["yMax"]))
    return total


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("out_dir"); ap.add_argument("tifs", nargs="+")
    ap.add_argument("--zoom", default="6,14"); ap.add_argument("--workers", type=int, default=2); ap.add_argument("--bbox")
    a = ap.parse_args()
    zmin, zmax = map(int, a.zoom.split(","))
    bbox = tuple(map(float, a.bbox.split(","))) if a.bbox else None
    total = {}
    for r in run(a.tifs, a.out_dir, zmin, zmax, a.workers, bbox):
        merge_bounds(total, r["bounds"])
        print(json.dumps({k: (len(v) if k == "tiles" else v) for k, v in r.items() if k != "bounds"}, ensure_ascii=False))
    print("tile bounds:", json.dumps(total))

In [ ]:
%%writefile /content/work/danfront_funcs.py
# ---- helper functions (pure Python; no Colab-specific calls) ----
import csv, json, math, os, shutil, subprocess, sys, time
import requests

def free_gb(path):
    return shutil.disk_usage(path).free / 1e9

def wait_for_disk(path, min_free=12, resume_at=20):
    """Pause while the local disk is nearly full (Drive needs time to sync its write cache)."""
    if free_gb(path) >= min_free:
        return
    print(f"  local disk low ({free_gb(path):.0f} GB free) - waiting for Drive to catch up ...")
    while free_gb(path) < resume_at:
        time.sleep(30)
    print("  continuing")

def download_parts(url, dest_prefix, part_size=4 * 1024**3, chunk=8 * 1024**2, disk_path="/content"):
    """Download url as raw volumes dest_prefix.001, .002, ... (7-Zip opens these directly).
    Finished parts are skipped, so the cell can be re-run with a fresh link after an interruption."""
    s = requests.Session()
    r = s.get(url, headers={"Range": "bytes=0-0"}, stream=True, timeout=60)
    cr = r.headers.get("Content-Range"); status = r.status_code; ctype = r.headers.get("Content-Type", ""); r.close()
    if status in (401, 403) or "text/html" in ctype:
        raise RuntimeError(f"The link was refused (HTTP {status}, {ctype}). It has probably expired: copy a fresh download link and run this cell again.")
    if status != 206 or not cr:
        raise RuntimeError(f"The server did not accept a ranged request (HTTP {status}). Send this message to Claude.")
    total = int(cr.split("/")[-1])
    size_file = dest_prefix + ".size"
    if os.path.exists(size_file):
        old = int(open(size_file).read().strip())
        if old != total:
            raise RuntimeError(f"This link points to a file of {total} bytes, but the parts already downloaded belong to a file of {old} bytes. Wrong link?")
    else:
        open(size_file, "w").write(str(total))
    nparts = math.ceil(total / part_size)
    print(f"archive size {total/1e9:.1f} GB -> {nparts} parts")
    t0 = time.time(); done_bytes = 0; new_bytes = 0
    for k in range(nparts):
        start = k * part_size; end = min(total, start + part_size) - 1; want = end - start + 1
        part = f"{dest_prefix}.{k+1:03d}"
        if os.path.exists(part) and os.path.getsize(part) == want:
            done_bytes += want; continue
        tmp = part + ".tmp"
        for attempt in range(5):
            try:
                r = s.get(url, headers={"Range": f"bytes={start}-{end}"}, stream=True, timeout=120)
                if r.status_code != 206:
                    raise RuntimeError(f"HTTP {r.status_code}")
                got = 0
                with open(tmp, "wb") as f:
                    for c in r.iter_content(chunk):
                        f.write(c); got += len(c)
                        wait_for_disk(disk_path)
                if got != want:
                    raise IOError(f"got {got} of {want} bytes")
                os.replace(tmp, part); break
            except Exception as e:
                print(f"  part {k+1}: {e} - retrying ({attempt+1}/5)"); time.sleep(20)
        else:
            raise RuntimeError("Download keeps failing. The link has probably expired: copy a fresh download link and run this cell again (finished parts are kept).")
        done_bytes += want; new_bytes += want
        rate = new_bytes / max(1, time.time() - t0) / 1e6
        print(f"part {k+1}/{nparts} done  ({done_bytes/1e9:.1f} of {total/1e9:.1f} GB, {rate:.0f} MB/s)")
    have = sum(os.path.getsize(f"{dest_prefix}.{k+1:03d}") for k in range(nparts))
    assert have == total, f"size mismatch: {have} != {total}"
    print("download complete and verified:", f"{total/1e9:.1f} GB")

def list_archive(first_volume, exe="7z"):
    """Return (properties dict, [(path, size)] of GeoTIFFs with coordinates) from the archive listing."""
    out = subprocess.run([exe, "l", "-slt", first_volume], capture_output=True, text=True)
    if out.returncode != 0:
        raise RuntimeError("7z could not read the archive:\n" + out.stdout[-2000:] + out.stderr[-2000:])
    head, _, body = out.stdout.partition("----------")
    props = dict(l.split(" = ", 1) for l in head.splitlines() if " = " in l)
    sheets = []
    for block in body.split("\n\n"):
        e = dict(l.split(" = ", 1) for l in block.splitlines() if " = " in l)
        p = e.get("Path", "")
        if not p.lower().endswith((".tif", ".tiff")) or e.get("Folder") == "+" or e.get("Attributes", "").startswith("D"):
            continue
        if "/raw/" in "/" + p.replace("\\", "/"):
            continue                      # RAW sheets (no coordinates) are not for this notebook
        sheets.append((p, int(e.get("Size", 0))))
    return props, sorted(sheets)

def load_done(done_csv):
    if not os.path.exists(done_csv):
        return {}
    with open(done_csv, newline="", encoding="utf-8") as f:
        return {r["path"]: r for r in csv.DictReader(f)}

DONE_FIELDS = ["path", "sheet", "status", "cell_i", "cell_j", "cell_coverage", "tiles"]

def run_batches(cfg, max_sheets=None, batch_size=12, bbox=None, workers=2):
    """Extract -> tile -> upload -> record, batch by batch. Safe to interrupt and re-run."""
    import tiles_2kf_batch as tb
    done = load_done(cfg["done_csv"])
    todo = [p for p, _ in cfg["sheets"] if p not in done]
    if max_sheets:
        todo = todo[:max_sheets]
    print(f"{len(done)} sheets already done, {len(todo)} to do in this run")
    if done and not os.path.exists(cfg["restored_marker"]):
        print("new session: fetching the tiles made so far back from R2 (needed to merge sheet edges) ...")
        rc = subprocess.run(["rclone", "copy", cfg["remote"], cfg["tiles"], "--transfers", "64", "--checkers", "64"]).returncode
        if rc != 0:
            raise RuntimeError("could not restore tiles from R2")
    os.makedirs(cfg["tiles"], exist_ok=True); open(cfg["restored_marker"], "w").close()
    bounds = json.load(open(cfg["bounds_json"])) if os.path.exists(cfg["bounds_json"]) else {}
    bounds = {int(z): b for z, b in bounds.items()}
    t0 = time.time(); n_done = 0
    for b0 in range(0, len(todo), batch_size):
        batch = todo[b0:b0 + batch_size]
        wait_for_disk(cfg["disk_path"])
        shutil.rmtree(cfg["extract"], ignore_errors=True); os.makedirs(cfg["extract"])
        lst = os.path.join(cfg["local"], "batch.txt")
        open(lst, "w", encoding="utf-8").write("\n".join(batch) + "\n")
        ex = subprocess.run([cfg.get("exe", "7z"), "x", cfg["first_volume"], f"-o{cfg['extract']}", f"-i@{lst}", "-y", "-bso0", "-bsp0"],
                            capture_output=True, text=True)
        if ex.returncode not in (0, 1):
            raise RuntimeError("7z extraction failed:\n" + ex.stdout[-1500:] + ex.stderr[-1500:])
        tifs, missing = [], []
        for p in batch:
            fp = os.path.join(cfg["extract"], p.replace("\\", "/"))
            (tifs if os.path.exists(fp) else missing).append((p, fp))
        results = tb.run([fp for _, fp in tifs], cfg["tiles"], 6, 14, workers=workers, bbox=bbox)
        changed = os.path.join(os.path.dirname(os.path.abspath(cfg["tiles"])), "changed_tiles.txt")
        if os.path.getsize(changed) > 0:
            for attempt in range(3):
                rc = subprocess.run(["rclone", "copy", cfg["tiles"], cfg["remote"], "--files-from", changed, "--no-traverse",
                                     "--transfers", "32"]).returncode
                if rc == 0:
                    break
                print("  upload failed, retrying ..."); time.sleep(15)
            else:
                raise RuntimeError("Upload to R2 failed 3 times. Check the R2 keys, then run this cell again.")
        rows = []
        for (p, _), r in zip(tifs, results):
            tb.merge_bounds(bounds, r["bounds"])
            c = r.get("cell", ["", ""])
            rows.append(dict(path=p, sheet=r["sheet"], status=r["status"], cell_i=c[0], cell_j=c[1],
                             cell_coverage=r.get("cell_coverage", ""), tiles=len(r["tiles"])))
        for p, _ in missing:
            rows.append(dict(path=p, sheet=os.path.basename(p), status="ERROR: not extracted", cell_i="", cell_j="", cell_coverage="", tiles=0))
        new_file = not os.path.exists(cfg["done_csv"])
        with open(cfg["done_csv"], "a", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=DONE_FIELDS)
            if new_file:
                w.writeheader()
            w.writerows(rows)
        json.dump(bounds, open(cfg["bounds_json"], "w"))
        shutil.rmtree(cfg["extract"], ignore_errors=True)
        n_done += len(batch)
        el = time.time() - t0; left = (len(todo) - n_done) * el / n_done
        bad = [r["sheet"] + ": " + r["status"] for r in rows if not r["status"].startswith("ok")]
        print(f"[{n_done}/{len(todo)}] batch done in total {el/60:.0f} min, about {left/60:.0f} min left" + (f"   notes: {bad}" if bad else ""))
    print("run finished")

def stitch_preview(tiles_dir, z, max_px=1600):
    """Stitch all local tiles of zoom z into one image (for a quick look)."""
    import glob
    from PIL import Image
    tiles = glob.glob(os.path.join(tiles_dir, str(z), "*", "*.png"))
    if not tiles:
        return None
    key = lambda p: (int(p.split(os.sep)[-2]), int(os.path.basename(p)[:-4]))
    xs = sorted({key(p)[0] for p in tiles}); ys = sorted({key(p)[1] for p in tiles})
    W = Image.new("RGBA", (256 * (xs[-1] - xs[0] + 1), 256 * (ys[-1] - ys[0] + 1)), (225, 225, 225, 255))
    for p in tiles:
        x, y = key(p); W.alpha_composite(Image.open(p).convert("RGBA"), ((x - xs[0]) * 256, (y - ys[0]) * 256))
    W = W.convert("RGB")
    if max(W.size) > max_px:
        W.thumbnail((max_px, max_px))
    return W

def summary(cfg):
    done = load_done(cfg["done_csv"])
    from collections import Counter
    c = Counter(r["status"].split(":")[0] for r in done.values())
    print("sheets:", len(done), dict(c))
    for r in done.values():
        if not r["status"].startswith("ok") or "irregular" in r["status"]:
            print("  ", r["sheet"], "-", r["status"], "coverage", r["cell_coverage"])
    b = json.load(open(cfg["bounds_json"]))
    print("\n--- copy everything below this line to Claude ---")
    print("sheets_ok:", sum(1 for r in done.values() if r["status"].startswith("ok")), "of", len(done))
    print("var kf2TileBounds = {")
    for z in sorted(b, key=int):
        v = b[z]; print(f"    {int(z)}: {{ xMin: {v['xMin']}, xMax: {v['xMax']}, yMin: {v['yMin']}, yMax: {v['yMax']} }},")
    print("};")

In [ ]:
import sys, importlib
sys.path.insert(0, LOCAL)
import danfront_funcs as F, tiles_2kf_batch
importlib.reload(F); importlib.reload(tiles_2kf_batch)
print('scripts loaded')

## 3. Connect to R2
Reads the three secrets and checks that the bucket can be reached. It should list a few tile folders (the Budapest test sheet is already there).

In [ ]:
os.environ.update({
    'RCLONE_CONFIG_R2_TYPE': 's3',
    'RCLONE_CONFIG_R2_PROVIDER': 'Cloudflare',
    'RCLONE_CONFIG_R2_ACCESS_KEY_ID': userdata.get('R2_ACCESS_KEY_ID').strip(),
    'RCLONE_CONFIG_R2_SECRET_ACCESS_KEY': userdata.get('R2_SECRET_ACCESS_KEY').strip(),
    'RCLONE_CONFIG_R2_ENDPOINT': userdata.get('R2_ENDPOINT').strip(),
    'RCLONE_S3_NO_CHECK_BUCKET': 'true',
})
REMOTE = 'r2:danfront/tiles-2kf'
import subprocess
r = subprocess.run(['rclone', 'lsf', REMOTE + '/'], capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError('R2 could not be reached. Check the three secrets (and that Notebook access is on).\n' + r.stderr[-800:])
print('R2 connection OK. Folders in tiles-2kf:', r.stdout.split())

## 4. Download the archive into Drive (1-2 hours, only once)

Get the real download address: in the browser start downloading `Geotif_export.7z` from SharePoint, open
`chrome://downloads`, right-click that download > *Copy link address*, then cancel the download. Paste the address below.

The link is only valid for a few hours. If the cell stops with "link expired", copy a fresh link and run the cell again:
it continues where it stopped. When the archive is complete this cell finishes in a second, so it is safe to re-run.

In [ ]:
URL = "PASTE_THE_DOWNLOAD_LINK_HERE"

ARCHIVE = f'{WORK}/Geotif_export.7z'          # saved as Geotif_export.7z.001, .002, ... (7-Zip reads these as one file)
size_file = ARCHIVE + '.size'
complete = False
if os.path.exists(size_file):
    import glob
    complete = sum(os.path.getsize(p) for p in glob.glob(ARCHIVE + '.[0-9][0-9][0-9]')) == int(open(size_file).read())
if complete:
    print('archive already complete on Drive - nothing to download')
else:
    F.download_parts(URL, ARCHIVE)

## 5. Look inside the archive
Lists the sheets and shows how many are already done.

In [ ]:
props, SHEETS = F.list_archive(ARCHIVE + '.001')
print({k: props[k] for k in ('Method', 'Solid', 'Blocks') if k in props})
print(len(SHEETS), 'sheets with coordinates,', round(sum(s for _, s in SHEETS) / 1e9, 1), 'GB')
print('first ones:', [p for p, _ in SHEETS[:5]])

CFG = dict(sheets=SHEETS, first_volume=ARCHIVE + '.001', local=LOCAL, extract=f'{LOCAL}/extract', tiles=f'{LOCAL}/tiles',
           disk_path='/content', done_csv=f'{WORK}/done.csv', bounds_json=f'{WORK}/tile_bounds.json',
           restored_marker=f'{LOCAL}/.restored', remote=REMOTE)
print(len(F.load_done(CFG['done_csv'])), 'sheets already processed')

## 6. Test with a few sheets (about 5 minutes)

Processes only the first 6 sheets and shows them stitched together. Check that the sheets join without title strips
or legends between them. If something looks wrong, stop here and send the picture and the messages to Claude.

In [ ]:
F.run_batches(CFG, max_sheets=6, batch_size=6)
from IPython.display import display
display(F.stitch_preview(CFG['tiles'], 11))

## 7. Process everything (several hours)

Safe to interrupt and run again. To process only part of the country, set `BBOX = (lon_min, lat_min, lon_max, lat_max)`.

In [ ]:
BBOX = None        # e.g. (18.0, 44.0, 23.5, 47.0)
F.run_batches(CFG, batch_size=12, bbox=BBOX)

## 8. Result
Overview picture and the summary. Copy the text below the line and send it to Claude: it is needed to switch the full layer on in `maps.html`.

In [ ]:
display(F.stitch_preview(CFG['tiles'], 8))
F.summary(CFG)